# Spello Model Training
### Domain-Specific Spell Correction Model for McCray Historical Documents

This notebook:
1. Fetches Wikipedia content for domain vocabulary
2. Builds sentence corpus and named entity whitelist
3. Trains conservative Spello model optimized for OCR correction
4. Evaluates model performance on test cases
5. Saves versioned model with metadata

In [2]:
# Import Required Libraries
import pandas as pd
import numpy as np
import re
import requests
from bs4 import BeautifulSoup
import time
import json
from collections import Counter
from datetime import datetime
import os
from spello.model import SpellCorrectionModel
import warnings
warnings.filterwarnings('ignore')

In [3]:

# Model versioning
MODEL_VERSION = "v1.0"
TRAINING_DATE = datetime.now().strftime("%Y-%m-%d_%H-%M")
MODEL_NAME = f"mccray_domain_spello_{MODEL_VERSION}_{TRAINING_DATE}"

print(f"Training Spello Model: {MODEL_NAME}")
print(f"Training started: {datetime.now()}")

Training Spello Model: mccray_domain_spello_v1.0_2025-10-05_20-26
Training started: 2025-10-05 20:26:51.747859


In [ ]:
# Configuration
WIKI_PAGES = [
    "https://en.wikipedia.org/wiki/John_Henry_McCray",
    "https://en.wikipedia.org/wiki/Lighthouse_and_Informer", 
    "https://en.wikipedia.org/wiki/Jenkins_Orphanage",
    "https://en.wikipedia.org/wiki/Southern_United_States",
    "https://en.wikipedia.org/wiki/Deep_South",
    "https://en.wikipedia.org/wiki/Savannah,_Georgia",
    "https://en.wikipedia.org/wiki/Charleston,_South_Carolina",
    "https://en.wikipedia.org/wiki/South_Carolina",
    "https://en.wikipedia.org/wiki/African-American_newspapers",
    "https://en.wikipedia.org/wiki/Pittsburgh_Courier",
    "https://en.wikipedia.org/wiki/Progressive_Democratic_Party_(South_Carolina)",
    "https://en.wikipedia.org/wiki/North_Carolina_Mutual_Life_Insurance_Company",
    "https://en.wikipedia.org/wiki/Jim_Crow_laws",
    "https://en.wikipedia.org/wiki/Civil_rights_movement",
    "https://en.wikipedia.org/wiki/History_of_South_Carolina",
    "https://en.wikipedia.org/wiki/Columbia,_South_Carolina",
    "https://en.wikipedia.org/wiki/Charleston_County,_South_Carolina",
    "https://en.wikipedia.org/wiki/North_Charleston,_South_Carolina",
    "https://en.wikipedia.org/wiki/Richland_County,_South_Carolina",
    "https://en.wikipedia.org/wiki/North_Carolina",
    "https://en.wikipedia.org/wiki/Raleigh,_North_Carolina",
    "https://en.wikipedia.org/wiki/Reconstruction_era",
    "https://en.wikipedia.org/wiki/Literacy_test",
    "https://en.wikipedia.org/wiki/Black_Codes_(United_States)",
    "https://en.wikipedia.org/wiki/Civil_Rights_Act_of_1964",
    "https://en.wikipedia.org/wiki/Harry_S._Truman",
    "https://en.wikipedia.org/wiki/NAACP",
    "https://en.wikipedia.org/wiki/W._E._B._Du_Bois",
    "https://en.wikipedia.org/wiki/Black_Reconstruction_in_America",
    "https://en.wikipedia.org/wiki/Ida_B._Wells",
    "https://en.wikipedia.org/wiki/Racial_segregation_in_the_United_States",
    "https://en.wikipedia.org/wiki/Plessy_v._Ferguson",
    "https://en.wikipedia.org/wiki/Racial_discrimination"
]

# Training parameters
TRAINING_CONFIG = {
    'min_length_for_spellcorrection': 4,
    'sentence_min_words': 5,
    'sentence_max_words': 50,
    'edit_distance_map': {
        4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2,
        11: 3, 12: 3, 13: 3, 14: 3, 15: 3
    }
}

print(f"Will fetch {len(WIKI_PAGES)} Wikipedia pages")
print(f"Training config: {TRAINING_CONFIG}")

Will fetch 33 Wikipedia pages
Training config: {'min_length_for_spellcorrection': 4, 'sentence_min_words': 5, 'sentence_max_words': 50, 'edit_distance_map': {4: 1, 5: 1, 6: 2, 7: 2, 8: 2, 9: 2, 10: 2, 11: 3, 12: 3, 13: 3, 14: 3, 15: 3}}


In [8]:
# Wikipedia MediaWiki API Functions - Enhanced Error Handling
import urllib.parse

def fetch_wiki_page_content(page_title):
    """
    Fetch Wikipedia page content using MediaWiki API
    Returns clean text content from the page with enhanced error handling
    """
    base_url = "https://en.wikipedia.org/w/api.php"
    
    # Clean and encode the page title properly
    clean_title = page_title.strip().replace('_', ' ')
    
    params = {
        'action': 'query',
        'format': 'json',
        'titles': clean_title,
        'prop': 'extracts',
        'exintro': False,
        'explaintext': True,
        'exsectionformat': 'plain',
        'exlimit': 1
    }
    
    try:
        # Add headers to appear more like a legitimate browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        response = requests.get(base_url, params=params, headers=headers, timeout=30)
        response.raise_for_status()  # Raise exception for bad status codes
        
        data = response.json()
        
        if 'query' not in data:
            print(f"No query results for: {clean_title}")
            return ""
        
        pages = data['query']['pages']
        page_id = list(pages.keys())[0]
        
        if page_id == '-1':
            print(f"Page not found: {clean_title}")
            return ""
        
        page_data = pages[page_id]
        content = page_data.get('extract', '')
        
        if not content:
            print(f"No content extracted for: {clean_title}")
            return ""
        
        # Basic content validation
        if len(content) < 100:
            print(f"Content too short for: {clean_title} ({len(content)} chars)")
            return ""
            
        print(f"Successfully fetched: {clean_title} ({len(content):,} chars)")
        return content
        
    except requests.exceptions.Timeout:
        print(f"Timeout fetching: {clean_title}")
        return ""
    except requests.exceptions.RequestException as e:
        print(f"Request error fetching {clean_title}: {e}")
        return ""
    except json.JSONDecodeError as e:
        print(f"JSON decode error for {clean_title}: {e}")
        return ""
    except Exception as e:
        print(f"Unexpected error fetching {clean_title}: {e}")
        return ""

def fetch_all_wiki_content(wiki_urls, max_retries=2):
    """
    Fetch content from all Wikipedia URLs with retry logic
    Returns combined text and individual page contents with detailed metrics
    """
    all_content = []
    page_contents = {}
    fetch_stats = {
        'successful': 0, 
        'failed': 0, 
        'total_chars': 0,
        'retries_used': 0,
        'failed_pages': []
    }
    
    print(f"Fetching {len(wiki_urls)} Wikipedia pages...")
    print("=" * 60)
    
    for i, url in enumerate(wiki_urls, 1):
        # Extract page title from URL more robustly
        if '/wiki/' in url:
            page_title = url.split('/wiki/')[-1]
        else:
            page_title = url.split('/')[-1]
        
        page_title = urllib.parse.unquote(page_title).replace('_', ' ')
        
        print(f"[{i}/{len(wiki_urls)}] Fetching: {page_title}")
        
        # Retry logic
        content = ""
        for attempt in range(max_retries + 1):
            if attempt > 0:
                fetch_stats['retries_used'] += 1
                print(f"  Retry {attempt} for: {page_title}")
                time.sleep(2)  # Longer wait on retry
            
            content = fetch_wiki_page_content(page_title)
            if content:
                break
        
        if content:
            all_content.append(content)
            page_contents[page_title] = content
            fetch_stats['successful'] += 1
            fetch_stats['total_chars'] += len(content)
        else:
            fetch_stats['failed'] += 1
            fetch_stats['failed_pages'].append(page_title)
            print(f"  FAILED: {page_title}")
        
        # Be respectful to Wikipedia servers
        time.sleep(1.5)
    
    combined_text = '\n\n'.join(all_content)
    
    print("\n" + "=" * 60)
    print(f"FETCH RESULTS:")
    print(f"  Successful: {fetch_stats['successful']}/{len(wiki_urls)} pages")
    print(f"  Failed: {fetch_stats['failed']} pages")
    print(f"  Retries used: {fetch_stats['retries_used']}")
    print(f"  Total characters: {fetch_stats['total_chars']:,}")
    
    if fetch_stats['failed_pages']:
        print(f"\nFailed pages:")
        for page in fetch_stats['failed_pages'][:10]:  # Show first 10 failures
            print(f"  - {page}")
        if len(fetch_stats['failed_pages']) > 10:
            print(f"  ... and {len(fetch_stats['failed_pages']) - 10} more")
    
    if fetch_stats['successful'] == 0:
        print("\nWARNING: No pages fetched successfully!")
        print("This will likely cause training issues.")
        
        # Fallback to a minimal dataset
        print("Creating minimal fallback corpus...")
        fallback_content = """
        John Henry McCray was an African American journalist and civil rights activist.
        He published the Lighthouse and Informer newspaper in Charleston, South Carolina.
        McCray was involved in the Progressive Democratic Party of South Carolina.
        He worked to advance civil rights and voting rights for African Americans.
        Charleston is a historic city in South Carolina.
        South Carolina is a state in the Deep South region of the United States.
        The civil rights movement fought against racial segregation and discrimination.
        """
        return fallback_content, {'Fallback Content': fallback_content}, {
            'successful': 1, 'failed': len(wiki_urls), 'total_chars': len(fallback_content)
        }
    
    return combined_text, page_contents, fetch_stats

# Fetch Wikipedia content with enhanced error handling
print("Fetching Wikipedia content with enhanced error handling...")
wiki_text, individual_pages, wiki_stats = fetch_all_wiki_content(WIKI_PAGES)

Fetching Wikipedia content with enhanced error handling...
Fetching 33 Wikipedia pages...
[1/33] Fetching: John Henry McCray
Successfully fetched: John Henry McCray (656 chars)
Successfully fetched: John Henry McCray (656 chars)
[2/33] Fetching: Lighthouse and Informer
[2/33] Fetching: Lighthouse and Informer
Successfully fetched: Lighthouse and Informer (671 chars)
Successfully fetched: Lighthouse and Informer (671 chars)
[3/33] Fetching: Jenkins Orphanage
[3/33] Fetching: Jenkins Orphanage
Successfully fetched: Jenkins Orphanage (676 chars)
Successfully fetched: Jenkins Orphanage (676 chars)
[4/33] Fetching: Southern United States
[4/33] Fetching: Southern United States
Successfully fetched: Southern United States (3,609 chars)
Successfully fetched: Southern United States (3,609 chars)
[5/33] Fetching: Deep South
[5/33] Fetching: Deep South
Successfully fetched: Deep South (1,400 chars)
Successfully fetched: Deep South (1,400 chars)
[6/33] Fetching: Savannah, Georgia
[6/33] Fetching:

In [ ]:
# Domain Vocabulary and Named Entity Extraction
def extract_named_entities(text):
    """
    Extract potential named entities from text
    Returns set of entities with statistics
    """
    entities = set()
    entity_stats = {'capitalized': 0, 'multi_word': 0, 'years': 0, 'states': 0}
    
    if not text or len(text) < 50:  # Skip very short text
        return entities, entity_stats
    
    # Capitalized words (potential proper nouns)
    capitalized_words = re.findall(r'\b[A-Z][a-z]+\b', text)
    entities.update(capitalized_words)
    entity_stats['capitalized'] = len(set(capitalized_words))
    
    # Multi-word capitalized phrases
    multi_word_entities = re.findall(r'\b(?:[A-Z][a-z]+\s+){1,3}[A-Z][a-z]+\b', text)
    entities.update(multi_word_entities)
    entity_stats['multi_word'] = len(set(multi_word_entities))
    
    # Years (1800-2023)
    years = re.findall(r'\b(?:18|19|20)\d{2}\b', text)
    entities.update(years)
    entity_stats['years'] = len(set(years))
    
    # Southern states and locations
    southern_pattern = r'\b(?:South Carolina|North Carolina|Georgia|Alabama|Mississippi|Louisiana|Tennessee|Arkansas|Virginia|Kentucky|Florida|Texas|Charleston|Savannah|Columbia|Atlanta|Richmond)\b'
    southern_terms = re.findall(southern_pattern, text, re.IGNORECASE)
    entities.update([term.title() for term in southern_terms])
    entity_stats['states'] = len(set(southern_terms))
    
    return entities, entity_stats

def build_sentence_corpus(wiki_text, min_words=5, max_words=50):
    """
    Build sentence corpus for spello training with quality metrics
    Enhanced to handle limited input data
    """
    if not wiki_text or len(wiki_text) < 100:
        print("WARNING: Very limited wiki text available for corpus building")
        return [], {'total': 0, 'too_short': 0, 'too_long': 0, 'accepted': 0}
    
    # Clean the text
    clean_text = re.sub(r'\n+', ' ', wiki_text)
    clean_text = re.sub(r'\s+', ' ', clean_text)
    clean_text = clean_text.strip()
    
    # Split into sentences using multiple delimiters
    sentences = re.split(r'[.!?]+\s+', clean_text)
    
    # Also try splitting on newlines if we don't have enough sentences
    if len(sentences) < 20:
        additional_sentences = re.split(r'\n+', clean_text)
        sentences.extend(additional_sentences)
    
    # Filter and collect stats
    filtered_sentences = []
    corpus_stats = {'total': len(sentences), 'too_short': 0, 'too_long': 0, 'accepted': 0}
    
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
            
        word_count = len(sent.split())
        
        if word_count < min_words:
            corpus_stats['too_short'] += 1
        elif word_count > max_words:
            corpus_stats['too_long'] += 1
            # Still include long sentences but truncate them
            words = sent.split()[:max_words]
            truncated = ' '.join(words)
            filtered_sentences.append(truncated)
            corpus_stats['accepted'] += 1
        else:
            filtered_sentences.append(sent)
            corpus_stats['accepted'] += 1
    
    # If we still don't have enough sentences, create some additional ones
    if len(filtered_sentences) < 10:
        print("wARNING..... DID NOT HAVE ENOUGH SENTENCES")
        
        for supp_sent in supplementary:
            if len(supp_sent.split()) >= min_words:
                filtered_sentences.append(supp_sent)
                corpus_stats['accepted'] += 1
    
    return filtered_sentences, corpus_stats

def create_mccray_whitelist():
    """
    Create comprehensive whitelist of protected terms
    Enhanced with more domain-specific terms
    """
    whitelist = {
        # Key figures
        'McCray', 'John', 'Henry', 'Lighthouse', 'Informer',
        
        # Geographic locations
        'Charleston', 'Savannah', 'Columbia', 'Georgia', 'Carolina',
        'Atlanta', 'Richmond', 'Virginia', 'Alabama', 'Mississippi',
        'Louisiana', 'Tennessee', 'Arkansas', 'Kentucky', 'Florida',
        'Texas', 'Carolinas', 'Talladega', 'Orangeburg', 'Beaufort',
        'Greenville', 'Spartanburg', 'Florence', 'Sumter', 'Anderson',
        
        # Organizations and institutions
        'NAACP', 'CORE', 'SCLC', 'SNCC', 'Progressive', 'Democratic',
        'Jenkins', 'Orphanage', 'Courier', 'Pittsburgh', 'Mutual',
        
        # Historical terms
        'Jim', 'Crow', 'segregation', 'desegregation', 'civil', 'rights',
        'Negro', 'colored', 'African', 'American', 'reconstruction',
        
        # Titles and initials
        'Dr', 'Mr', 'Mrs', 'Ms', 'Jr', 'Sr', 'Rev', 'Prof',
        
        # Decades and years
        '1940', '1950', '1960', '1970', '1980', '1930s', '1940s', '1950s',
        '1960s', '1970s', '1930', '1931', '1932', '1933', '1934', '1935'
    }
    
    return whitelist

# Build training corpus with enhanced error handling
print("\nBuilding training corpus...")

if not wiki_text or len(wiki_text) < 100:
    print("WARNING: Insufficient Wikipedia content fetched!")
    print("Using fallback content for training...")
    
sentences, sentence_stats = build_sentence_corpus(
    wiki_text, 
    TRAINING_CONFIG['sentence_min_words'],
    TRAINING_CONFIG['sentence_max_words']
)

print("\nExtracting named entities...")
all_entities = set()
total_entity_stats = {'capitalized': 0, 'multi_word': 0, 'years': 0, 'states': 0}

# Extract entities from fetched pages
for page_title, content in individual_pages.items():
    entities, stats = extract_named_entities(content)
    all_entities.update(entities)
    for key in total_entity_stats:
        total_entity_stats[key] += stats[key]

# Add manual whitelist (always include these)
manual_whitelist = create_mccray_whitelist()
all_entities.update(manual_whitelist)

print(f"\nCorpus Statistics:")
print(f"  Total sentences processed: {sentence_stats['total']}")
print(f"  Accepted for training: {sentence_stats['accepted']}")
print(f"  Too short (< {TRAINING_CONFIG['sentence_min_words']} words): {sentence_stats['too_short']}")
print(f"  Too long (> {TRAINING_CONFIG['sentence_max_words']} words): {sentence_stats['too_long']}")

print(f"\nEntity Statistics:")
print(f"  Total unique entities: {len(all_entities)}")
print(f"  Manual whitelist terms: {len(manual_whitelist)}")
print(f"  Extracted entities by type: {total_entity_stats}")

print(f"\nSample entities: {sorted(list(all_entities))[:15]}")
print(f"\nSample training sentences:")
for i, sent in enumerate(sentences[:3], 1):
    print(f"  {i}. {sent}")

# Validation check
if len(sentences) < 5:
    print("\nERROR: Insufficient training sentences!")
    print("This will likely cause training to fail.")
elif len(sentences) < 20:
    print("\nWARNING: Limited training sentences.")
    print("Model performance may be suboptimal.")
else:
    print(f"\nTraining corpus ready: {len(sentences)} sentences")


Building training corpus...

Extracting named entities...

Corpus Statistics:
  Total sentences processed: 523
  Accepted for training: 487
  Too short (< 5 words): 36
  Too long (> 50 words): 10

Entity Statistics:
  Total unique entities: 1058
  Manual whitelist terms: 76
  Extracted entities by type: {'capitalized': 1339, 'multi_word': 459, 'years': 140, 'states': 62}

Sample entities: ['1800', '1832', '1835', '1840', '1857', '1860', '1861', '1862', '1863', '1865', '1866', '1867', '1868', '1875', '1876']

Sample training sentences:
  1. John Henry McCray (1910–1987) was an American journalist, newspaper publisher, politician, civil rights activist, and college academic administrator
  2. An African American, he worked at some of the country's most prominent Black newspapers including the Lighthouse and Informer newspaper of South Carolina (from 1941 to 1954); the Charleston Messenger; the Pittsburgh Courier as the Carolina editor (from 1960 to 1962); the Baltimore Afro-American (fr

In [ ]:
# Model Training with Enhanced Error Handling
def train_domain_spello_model(sentences, entities_whitelist, config):
    """
    Train spello model with conservative OCR-friendly configuration
    Enhanced with error handling and minimum data validation
    """
    print("Validating training data...")
    
    # Validate we have sufficient data
    if len(sentences) < 10:
        print(f"WARNING: Only {len(sentences)} sentences available for training!")
        print("This may result in poor model performance.")
        
        # Add some basic sentences if we're really low on data
        if len(sentences) < 5:
            fallback_sentences = [
                "John Henry McCray was a journalist and civil rights activist.",
                "He published the Lighthouse and Informer newspaper in Charleston.",
                "McCray worked for civil rights in South Carolina.",
                "Charleston is located in South Carolina.",
                "The Progressive Democratic Party was active in the 1940s.",
                "African American newspapers covered civil rights issues.",
                "South Carolina is in the Deep South region.",
                "The civil rights movement fought segregation and discrimination.",
                "McCray advocated for voting rights and equal treatment.",
                "The Lighthouse and Informer was published regularly."
            ]
            sentences.extend(fallback_sentences)
            print(f"Added {len(fallback_sentences)} fallback sentences. Total: {len(sentences)}")
    
    print("Initializing Spello model...")
    sp = SpellCorrectionModel(language='en')
    
    print(f"Training on {len(sentences)} sentences...")
    print("Training sentences preview:")
    for i, sent in enumerate(sentences[:3], 1):
        print(f"  {i}. {sent[:80]}...")
    
    training_start = time.time()
    
    try:
        sp.train(sentences)
        training_time = time.time() - training_start
        print(f"Training completed successfully in {training_time:.2f} seconds")
        
    except Exception as e:
        print(f"Training failed with error: {e}")
        print("This might be due to insufficient or poor quality training data.")
    
    # Apply conservative configuration
    print("Configuring model for OCR correction...")
    sp.config.min_length_for_spellcorrection = config['min_length_for_spellcorrection']
    
    # Safely apply edit distance map
    try:
        sp.config.symspell_allowed_distance_map = config['edit_distance_map']
    except Exception as e:
        print(f"Warning: Could not apply edit distance map: {e}")
    
    # Store metadata
    sp.whitelist = entities_whitelist
    sp.training_metadata = {
        'version': MODEL_VERSION,
        'training_date': TRAINING_DATE,
        'sentence_count': len(sentences),
        'entity_count': len(entities_whitelist),
        'training_time_seconds': training_time,
        'config': config,
        'wiki_pages': len(WIKI_PAGES),
        'wiki_stats': wiki_stats
    }
    
    return sp

# Train the model with enhanced error handling
print("\n=== TRAINING SPELLO MODEL ===")
try:
    sp_model = train_domain_spello_model(sentences, all_entities, TRAINING_CONFIG)
    print(f"\nModel training complete!")
    print(f"Model metadata: {sp_model.training_metadata}")
    
    # Quick test to verify model works
    print("\nQuick model test:")
    test_word = "teh"
    try:
        correction = sp_model.spell_correct(test_word)
        print(f"  '{test_word}' -> '{correction}'")
    except Exception as e:
        print(f"  Model test failed: {e}")
        
except Exception as e:
    print(f"FATAL: Model training failed completely: {e}")
    print("Cannot continue without a trained model.")
    raise e


=== TRAINING SPELLO MODEL ===
Validating training data...
Initializing Spello model...
Training on 487 sentences...
Training sentences preview:
  1. John Henry McCray (1910–1987) was an American journalist, newspaper publisher, p...
  2. An African American, he worked at some of the country's most prominent Black new...
  3. McCray was a co-founder of the Progressive Democratic Party (PDP) of South Carol...
Spello training started..
Context model training started ...
Context model training started ...
Symspell training started ...
Symspell training started ...
Phoneme training started ...
Spello training completed successfully ...
Training completed successfully in 1.15 seconds
Configuring model for OCR correction...

Model training complete!
Model metadata: {'version': 'v1.0', 'training_date': '2025-10-05_20-26', 'sentence_count': 487, 'entity_count': 1058, 'training_time_seconds': 1.1499524116516113, 'config': {'min_length_for_spellcorrection': 4, 'sentence_min_words': 5, 'sentence_

In [16]:
# Model Evaluation on Test Cases
def create_ocr_test_cases():
    """
    Create comprehensive test cases for OCR error correction
    """
    test_cases = [
        # Name corrections
        {
            'input': 'John Heniy McCiay',
            'expected': 'John Henry McCray',
            'category': 'proper_names'
        },
        {
            'input': 'Dr. McCiay published',
            'expected': 'Dr. McCray published',
            'category': 'proper_names'
        },
        # Publication names
        {
            'input': 'Lighthouse and lnformer',
            'expected': 'Lighthouse and Informer',
            'category': 'publications'
        },
        # Geographic locations
        {
            'input': 'Charleston South Caiolina',
            'expected': 'Charleston South Carolina',
            'category': 'geography'
        },
        {
            'input': 'Savannah Geoigia',
            'expected': 'Savannah Georgia',
            'category': 'geography'
        },
        # Common words with dates
        {
            'input': 'The yeai 1950 was important',
            'expected': 'The year 1950 was important',
            'category': 'common_words'
        },
        {
            'input': 'publisned in 1945',
            'expected': 'published in 1945',
            'category': 'common_words'
        },
        # Civil rights terms
        {
            'input': 'civil rigj s movement',
            'expected': 'civil rights movement',
            'category': 'civil_rights'
        },
        # Mixed cases
        {
            'input': 'The Lighlhouse and lnformer newspape covered civil righ  in South Caiolina',
            'expected': 'The Lighthouse and Informer newspaper covered civil rights in South Carolina',
            'category': 'mixed'
        },
        # ACTUAL ORIGINAL TRANSCRIPTS
        {
'input':
"""
Sec. 2  THE COURIER  Sept. 9, 1961  EDITORIALS  Without Ballyhoo  Without any ballyhoo, picketing or mass meetings,  Negro newspaper publishers met recently in Chicago and  formed the Consolidated Publishers, Inc., by merging three  publishers' representatives' agencies to 
make a centralized  drive for a presently $7 million billing.  Previously there had been in the field, Associated  Publishers, Inc., Interstate United Newspapers, Inc., and  the Defender Publications, all competing for parts of the  lucrative advertising budget which keep 
the Negro newspapers running.  Negotiations went on for two years before an agreement was reached.   _  What this means is that the newly formed Consolidated Publishers, Inc., will enable the national advertiser to  reach over 1,000,000 Negro households in the prinicpal  
urban markets over the country with local contact, through  principal offices in New York, Chicago, Detroit, Los Angeles, and other big cities.  This important change fas forced by changed conditions in the advertising market, but the important thing is  that Negro newspaper
 publishers saw the need and rose to  the occasion in a quiet, intelligent manner, and they will  now control all of the advertising appearing in the Negro  press.  This is an evidence of what other related Negro business or organization should do for their own and the 
 group  advancement.  While competition is the life of trade, it can be furthered by cooperation and consolidation, which is just what  the publishers have done, quietly and effectively.  It is noteworthy in this connection that just over  20 years ago, all advertising 
 handled in Negro newspapers  of a national nature was in the hands of white concerns,  and that it was the late Robert L. Vann, founder of this  newspaper who then saw the need of Negroes handling  this vast business for themselves.  What could not be accomplished for the 
 economic advancement of the Negro community if other Negro concerns followed the example of the publishers?  Doing Us No Good  It was inevitable that Robert P. Williams, deposed  NAACP branch president of Monroe, N. C. should sow  what he reaped.  Mr. Williams was,
  and is, an extremist. His urging  of the use of force against jim crowism was the cause of  his ouster from the NAACP which must of necessity  operate on a basis of legalism.  Mr. Williams was already in the doghouse because of  his romance 
  with Fidel Castro of Cuba, his frequent host,  end the praise heaped upon him by the Bearded One.  This was bad enough, but to have organized an armed  force and kidnaped a white couple to compel compliance  with his program, was 
  just too much for constituted authority to accept.  To be sure, jim crowism is a great affliction anywhere,  especially in the South, but it cannot be cured by resorting to  crime.  It is putting it mildly to state that such 
  antics do our  cause no good.  Costly Freedom Rides  According to the Congress of Racial Equality, it has  spent to date $200,250 on Freedom Rides, including bail,  legal fees, legal expenses, travel, training, hospital bills, 
   phones and telegrams, and printing; and not including  overhead, office expenses and fund raising costs, and not  all bills have been received.  There are some who will contend that this expenditure was and is worthwhile, 
   but since nothing has been  solved by this activity, we are constrained to be skeptical.  Muslim Descent From the Summit  It is of considerable interest that the widely bally-  hooed Harlem mass meeting on Aug. 27 of the
    so-called  Black Muslims was a flop.  A huge armory was rented, great sums were spent on  advertising the appearance of the black supremacy leader,  Elijah Muhammad, and a minimum of 15,000 persons was  expected; but only about 5,000, 
    mostly Muslims, showed  up.  With over one and a quarter million Negroes in New  York City, this must be regarded as a disappointing turnout for the most publicized Negro organization of the last  decade, considering the hordes that 
    greeted the Negro  Elks at their New York convention.  The sad fact is that Harlemites were conspicuous by  their absence, and those that attended the Muslim meeting gathered outside and listened by loudspeaker.  
    Negro Americans, it is apparent, are not concerned  with queer, unreasonable, extremist organizations that  advance ridiculous programs of salvation, and they are  no more fetched by appeals to black supremacy than programs of white supremacy.  
    In short, black Ku Klux Klans are no more acceptable to Negroes than white Ku Klux Klans.  Views and Reviews  By George S. Schuyler  DESPITE THE current  gnashing of teeth and  frothing at the mouth over colonialism-imperialism 
    (spurred  by the Communists, who  have the largest, and most populous colonies of all), there is  every Indication that it will be  with us for  centuries as it  has been with  us for mlllen-  iums.  ÎÃÃ¶ The so-called backward  and newly-  
    emergent  "democracies"  in Asia and  Africa, along  with the rag-  t a g "republics" of Latin  American derivation, will not only continue  to be dependent upon the big  political power structures, but  will become more dependent.  Whether the free or communist powers prevail, the white  folks, for better or for worse,  will remain far in the lead, to  the great pain, one presumes,  of the militant, black, brown  and yellow nationalists who  yap in the U. N. corridors  and assembly halls.  Mr. Schuyler  WHY, DEAR brethren, is this  so?  Because (a) the underdeveloped nations (largely one-crop  economies just emerging from  subsistence agriculture) are  falling father behind economically, as the mechanized  nations forge farther ahead;  because (b) their populations  continue to swell, thus wiping  out the advantage of any gains  that are made because (c)  modern chemistry has already  made Western industry free of  reliance on the copper, tin, and  bauxite of tropical lands, and  there will be less and less importing of these resources, even  iron which is yielding to synthetics and plastics; because  (d) world supplies of cocoa, oil,  coffee and tea are in such sur.  plus that the price Is everywhere depressed, as is true of  sugar; because (e) it costs at  least an Investment of $600 for  each factory job desired, and  the money is only available  from the free or Communist  power structures, and it is their  bounty and goodwill that holds  the future of these weaker and  poorer countries in their grip.  Thus freedom becomes academic, like the freedom of the  man who is "bugged" by weekly installment payments.  ONE WONDERS how many  of the miseducated leaders of  the backward countries know  these facts of life of the modern world. Every one of these  countries is bankrupt or close  to it.  Having steamed up their few  literate followers to believing  that they have resources that  some body has to buy and  which will pay for their necessary education and industrialization, it must have dawned on  somebody has to buy and  become less indispensable yearly, as substitutes are created in  growing profusion by the great  industrial laboratories, even to  diamonds.  So the pretense that these  scores of weak and improv-  erished countries have the USA  and/or the USSR over the barrel, is a hollow one.  These countries will go on  having their flags, legislatures,  miniscule armies and Cadillac-  ed politicians, but to keep from  being tossed into the pot by  the hungry hordes they have  organized, they will have to  continue to get handouts from  those who will give them. If  this isn't colonialismÎÃÃ¶imperialism, what is it?  A New South Dawning . .    by f. l shuttiesworth  I BELIEVE THAT the South  has learned one lesson well.  That is, that violence, either  by mobs or policemen, will not  be allowed to frustrate the legitimate claims by Negroes of  their rights, no longer. Those  of us who have fought the civil  rights battle  (for years)  against all  odds ... from  frustrated officials to bellicose and enraged neighbors, from per-  secution by  law officers to  outright vici-  ousness by the  Klansmen . . .  have prayed  long for the  dawning of this day.  Always, we have known that  when the men in charge of the  city, hamlet or state, will array themselves, firmly, against  violence and declare their determination to enforce the  laws, the problems will be lessened, and solutions of any difficulties will be found, easily.  Shuttiesworth  Courier  The Pittsburgh Courier Publishing Company, Inc.  2628 Central Ave.  PittsburghTelephone:  MUseum 3-2000  SATURDAY, SEPTEMBER 9, 1961  ROBERT L VANH. EDITOR  MRS.   ROBERT  L.  VANN  president and treasurer  S.   B.  FULLER  Publisher  Chm. of Board  P.   L.   PRATTIS  Associate  Publisher,  Assnt. Treasurer.  WILLIAM A. NUNN   BR.  Editor  Second-rate Postage Paid at Pittsburgh,  Pa.  SUBSCRIPTION RATES  la Continenta   United States,  Hawaii, Alaska  Puerto Rico and Virgin Islands  SIX  MONTHS  $5.00   ... ONE YEAR  $10.00 ... TWO YEARS  IN CANADA  $4.28    SIX  MONTHS  $7.00 ONE YEAR  $12.00 TWO YEARS  FOREIGN  $8.25 SIX MONTHS  ONE YEAR  $18.00 TWO YEARS  The Plttahurgh Coirlar does not guarantee  either the use or return of photographs.  There would have been no  Little Rocks nor New Orleans,  had the Fauduses and Dav-  ises been men fitted for this  hour. But this is history . . .  we hope. And, we pray that the  future pages in this book will  reflect sanity, uprightness and  justice.  ÎÃ»Ã¡  THUS, ALL Americans can  take pride in the programs of  law enforcement, announced  for the Atlanta, Dallas and  Houston schools to be integrated this week. Crowds will not  be allowed to form, and people  with no business at the schools  will not be allowed to enter.  This is a positive step forward, and one in marked contrast to that where crowds  milled and molested everybody  . . . even police officers. The  outcome is clear: law and order will prevail, and the chaos  which, almost, has become traditional will be a thing of the  past.  This is not, by any maens, to  praise token integration . . .  for that is not real American  democracy. And American Negroes never must compromise  their rights, nor be satisfied  with crumbs when they, like  others, are entitled to the loaf.  But, since the integration ordered is token . . . and this is  nothing more than a start, then  it is good to see law officers  guarantee order and peace.  ÎÃ»Ã¡  IT IS MY prayer that the  rulers of Alabama and Birmingham will take lessons from  Mayor Hartsfield, Chief Jenkins and others who represent  progress and enlightenment.  The other side of the coin is  bad. The North Carolina episode, in which Robert Williams  is involved, is unfortunate. So  far as the civil-rights struggle  is concerned, it was wrong to  announce a policy of matching  violence with violence in the  first place. And to kidnap and  hold hostages ... if the press  were right ...   is a think unthinkable!  This played into the hands of  officials who, in the first place,  had no sympathy for the Negro  cause. And, now we read of a  Federal warrant for Mr. Williams' arrest. It may be that  these people were maneuvered  into a trap. We hope for the  best, here.  ÎÃ»Ã¡  JUST THE SAME, the Negro, in his quest for freedom,  MUST use non-violence as a  technique, at least, as the philosophy and need of this hourÎÃÃ¶  in truth. Negroes need the considerate and sympathetic understanding of mankind. Without resort to violence, he has  a better chance of obtaining  it . . . and his full freedom.  Courier Verse  Questions  If you were hungry and had  the cash,  But couldn't even purchase a  plate  of giblet hash,  Because you were beaten by a  Jim Crow lashÎÃÃ¶  Would you like it?  If you were born in a certain  place,  But just because of a colored  face,  You had to live in a special  spaceÎÃÃ¶  Would you like it?  If what you had built someone  tried to destroy,  If you were treated like a child  with a toy,  If strange people termed you  "Uncle" and "Boy"ÎÃÃ¶  Would you like it?  Just questions, America,    we  humbly ask;  Is giving democracy such   a  task?  If freedom and justice wear a  double maskÎÃÃ¶  Should we like it?  ÎÃÃ¶J. FARLEY RAGLAND construed as"
            
""",
    'expected': '',
    'category': 'mixed'
        }
    ]
    
    return test_cases

def evaluate_model_performance(model, test_cases):
    """
    Evaluate model performance on test cases
    """
    results = []
    category_stats = {}
    
    print("\n=== MODEL EVALUATION ===")
    
    for i, test_case in enumerate(test_cases, 1):
        input_text = test_case['input']
        expected = test_case['expected']
        category = test_case['category']
        
        try:
            result = model.spell_correct(input_text)
            corrected = result['spell_corrected_text'] if isinstance(result, dict) else str(result)
            
            # Calculate accuracy
            is_correct = corrected.lower() == expected.lower()
            
            # Word-level accuracy
            input_words = input_text.split()
            expected_words = expected.split()
            corrected_words = corrected.split()
            
            word_accuracy = 0
            if len(expected_words) == len(corrected_words):
                matches = sum(1 for e, c in zip(expected_words, corrected_words) 
                            if e.lower() == c.lower())
                word_accuracy = matches / len(expected_words)
            
            test_result = {
                'test_id': i,
                'category': category,
                'input': input_text,
                'expected': expected,
                'corrected': corrected,
                'exact_match': is_correct,
                'word_accuracy': word_accuracy
            }
            
            results.append(test_result)
            
            # Update category stats
            if category not in category_stats:
                category_stats[category] = {'total': 0, 'correct': 0}
            category_stats[category]['total'] += 1
            if is_correct:
                category_stats[category]['correct'] += 1
            
            print(f"Test {i} ({category}):")
            print(f"  Input:     '{input_text}'")
            print(f"  Expected:  '{expected}'")
            print(f"  Corrected: '{corrected}'")
            print(f"  Match: {is_correct}, Word Accuracy: {word_accuracy:.2f}")
            print()
            
        except Exception as e:
            print(f"Error in test {i}: {e}")
            results.append({
                'test_id': i,
                'category': category,
                'input': input_text,
                'expected': expected,
                'corrected': 'ERROR',
                'exact_match': False,
                'word_accuracy': 0.0
            })
    
    # Calculate overall statistics
    total_tests = len(results)
    exact_matches = sum(1 for r in results if r['exact_match'])
    avg_word_accuracy = np.mean([r['word_accuracy'] for r in results])
    
    evaluation_summary = {
        'total_tests': total_tests,
        'exact_matches': exact_matches,
        'exact_match_rate': exact_matches / total_tests if total_tests > 0 else 0,
        'average_word_accuracy': avg_word_accuracy,
        'category_performance': category_stats
    }
    
    return results, evaluation_summary

# Run evaluation
test_cases = create_ocr_test_cases()
eval_results, eval_summary = evaluate_model_performance(sp_model, test_cases)

print("\n=== EVALUATION SUMMARY ===")
print(f"Total tests: {eval_summary['total_tests']}")
print(f"Exact matches: {eval_summary['exact_matches']}/{eval_summary['total_tests']} ({eval_summary['exact_match_rate']:.1%})")
print(f"Average word accuracy: {eval_summary['average_word_accuracy']:.2%}")

print("\nCategory Performance:")
for category, stats in eval_summary['category_performance'].items():
    accuracy = stats['correct'] / stats['total'] if stats['total'] > 0 else 0
    print(f"  {category}: {stats['correct']}/{stats['total']} ({accuracy:.1%})")


=== MODEL EVALUATION ===
Test 1 (proper_names):
  Input:     'John Heniy McCiay'
  Expected:  'John Henry McCray'
  Corrected: 'John henry mccray'
  Match: True, Word Accuracy: 1.00

Test 2 (proper_names):
  Input:     'Dr. McCiay published'
  Expected:  'Dr. McCray published'
  Corrected: 'Dr  mccray published'
  Match: False, Word Accuracy: 0.67

Test 3 (publications):
  Input:     'Lighthouse and lnformer'
  Expected:  'Lighthouse and Informer'
  Corrected: 'Lighthouse and informer'
  Match: True, Word Accuracy: 1.00

Test 4 (geography):
  Input:     'Charleston South Caiolina'
  Expected:  'Charleston South Carolina'
  Corrected: 'Charleston South carolina'
  Match: True, Word Accuracy: 1.00

Test 5 (geography):
  Input:     'Savannah Geoigia'
  Expected:  'Savannah Georgia'
  Corrected: 'Savannah georgia'
  Match: True, Word Accuracy: 1.00

Test 6 (common_words):
  Input:     'The yeai 1950 was important'
  Expected:  'The year 1950 was important'
  Corrected: 'The year 1950 was 

In [ ]:
# Save Versioned Model and Metadata
def save_model_with_metadata(model, model_name, eval_results, eval_summary):
    """
    Save trained model with comprehensive metadata
    """
    # Create spello_models directory if it doesn't exist
    models_dir = "../spello_models"
    os.makedirs(models_dir, exist_ok=True)
    
    # Model path
    model_path = os.path.join(models_dir, model_name)
    
    # Save the spello model
    model.save(model_path)
    
    # Create comprehensive metadata
    metadata = {
        'model_info': {
            'name': model_name,
            'version': MODEL_VERSION,
            'training_date': TRAINING_DATE,
            'model_path': model_path
        },
        'training_data': {
            'wiki_pages_fetched': len(WIKI_PAGES),
            'wiki_pages_successful': wiki_stats['successful'],
            'total_characters': wiki_stats['total_chars'],
            'sentence_corpus_size': len(sentences),
            'named_entities_count': len(all_entities),
            'manual_whitelist_size': len(manual_whitelist)
        },
        'model_configuration': TRAINING_CONFIG,
        'evaluation_results': {
            'summary': eval_summary,
            'detailed_results': eval_results
        },
        'corpus_statistics': {
            'sentence_stats': sentence_stats,
            'entity_stats': total_entity_stats
        },
        'training_performance': {
            'training_time_seconds': model.training_metadata['training_time_seconds']
        }
    }
    
    # Save metadata as JSON
    metadata_path = f"{model_path}_metadata.json"
    with open(metadata_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)
    
    # Save evaluation results as CSV
    eval_df = pd.DataFrame(eval_results)
    eval_path = f"{model_path}_evaluation.csv"
    eval_df.to_csv(eval_path, index=False)
    
    # Save whitelist as text file
    whitelist_path = f"{model_path}_whitelist.txt"
    with open(whitelist_path, 'w', encoding='utf-8') as f:
        for entity in sorted(all_entities):
            f.write(f"{entity}\n")
    
    return {
        'model_path': model_path,
        'metadata_path': metadata_path,
        'evaluation_path': eval_path,
        'whitelist_path': whitelist_path
    }

# Save model and metadata
print("\n=== SAVING MODEL ===")
saved_files = save_model_with_metadata(sp_model, MODEL_NAME, eval_results, eval_summary)

print("Model saved successfully!")
print("Files created:")
for file_type, path in saved_files.items():
    print(f"  {file_type}: {path}")

# Create model registry entry
registry_path = "../spello_models/model_registry.json"
registry_entry = {
    MODEL_NAME: {
        'created': datetime.now().isoformat(),
        'version': MODEL_VERSION,
        'performance': {
            'exact_match_rate': eval_summary['exact_match_rate'],
            'word_accuracy': eval_summary['average_word_accuracy']
        },
        'files': saved_files
    }
}

# Update registry
if os.path.exists(registry_path):
    with open(registry_path, 'r') as f:
        registry = json.load(f)
else:
    registry = {}

registry.update(registry_entry)

with open(registry_path, 'w') as f:
    json.dump(registry, f, indent=2)

print(f"\nModel registered in: {registry_path}")


=== SAVING MODEL ===
Model saved successfully!
Files created:
  model_path: ../models\mccray_domain_spello_v1.0_2025-10-05_20-26
  metadata_path: ../models\mccray_domain_spello_v1.0_2025-10-05_20-26_metadata.json
  evaluation_path: ../models\mccray_domain_spello_v1.0_2025-10-05_20-26_evaluation.csv
  whitelist_path: ../models\mccray_domain_spello_v1.0_2025-10-05_20-26_whitelist.txt

Model registered in: ../models/model_registry.json


## Training Complete!

### Model Information:
- **Model Name**: `{MODEL_NAME}`
- **Version**: `{MODEL_VERSION}`
- **Training Date**: `{TRAINING_DATE}`

### Training Summary:
- Wikipedia pages fetched and processed
- Domain-specific sentence corpus created
- Named entity whitelist generated
- Model trained with OCR-optimized configuration
- Comprehensive evaluation completed
- Model saved with full metadata

### Next Steps:
1. Use `spello_apply.ipynb` to apply corrections to datasets
2. Review evaluation results in `{MODEL_NAME}_evaluation.csv`
3. Check model registry for version tracking
4. Monitor performance and retrain if needed

### Files Created:
- Trained model: `../models/{MODEL_NAME}/`
- Metadata: `../models/{MODEL_NAME}_metadata.json`
- Evaluation: `../models/{MODEL_NAME}_evaluation.csv`
- Whitelist: `../models/{MODEL_NAME}_whitelist.txt`
- Registry: `../models/model_registry.json`


In [17]:
check = '''
Sec. 2  THE COURIER  Sept. 9, 1961  EDITORIALS  Without Ballyhoo  Without any ballyhoo, picketing or mass meetings,  Negro newspaper publishers met recently in Chicago and  formed the Consolidated Publishers, Inc., by merging three  publishers' representatives' agencies to 
make a centralized  drive for a presently $7 million billing.  Previously there had been in the field, Associated  Publishers, Inc., Interstate United Newspapers, Inc., and  the Defender Publications, all competing for parts of the  lucrative advertising budget which keep 
the Negro newspapers running.  Negotiations went on for two years before an agreement was reached.   _  What this means is that the newly formed Consolidated Publishers, Inc., will enable the national advertiser to  reach over 1,000,000 Negro households in the prinicpal  
urban markets over the country with local contact, through  principal offices in New York, Chicago, Detroit, Los Angeles, and other big cities.  This important change fas forced by changed conditions in the advertising market, but the important thing is  that Negro newspaper
 publishers saw the need and rose to  the occasion in a quiet, intelligent manner, and they will  now control all of the advertising appearing in the Negro  press.  This is an evidence of what other related Negro business or organization should do for their own and the 
 group  advancement.  While competition is the life of trade, it can be furthered by cooperation and consolidation, which is just what  the publishers have done, quietly and effectively.  It is noteworthy in this connection that just over  20 years ago, all advertising 
 handled in Negro newspapers  of a national nature was in the hands of white concerns,  and that it was the late Robert L. Vann, founder of this  newspaper who then saw the need of Negroes handling  this vast business for themselves.  What could not be accomplished for the 
 economic advancement of the Negro community if other Negro concerns followed the example of the publishers?  Doing Us No Good  It was inevitable that Robert P. Williams, deposed  NAACP branch president of Monroe, N. C. should sow  what he reaped.  Mr. Williams was,
  and is, an extremist. His urging  of the use of force against jim crowism was the cause of  his ouster from the NAACP which must of necessity  operate on a basis of legalism.  Mr. Williams was already in the doghouse because of  his romance 
  with Fidel Castro of Cuba, his frequent host,  end the praise heaped upon him by the Bearded One.  This was bad enough, but to have organized an armed  force and kidnaped a white couple to compel compliance  with his program, was 
  just too much for constituted authority to accept.  To be sure, jim crowism is a great affliction anywhere,  especially in the South, but it cannot be cured by resorting to  crime.  It is putting it mildly to state that such 
  antics do our  cause no good.  Costly Freedom Rides  According to the Congress of Racial Equality, it has  spent to date $200,250 on Freedom Rides, including bail,  legal fees, legal expenses, travel, training, hospital bills, 
   phones and telegrams, and printing; and not including  overhead, office expenses and fund raising costs, and not  all bills have been received.  There are some who will contend that this expenditure was and is worthwhile, 
   but since nothing has been  solved by this activity, we are constrained to be skeptical.  Muslim Descent From the Summit  It is of considerable interest that the widely bally-  hooed Harlem mass meeting on Aug. 27 of the
    so-called  Black Muslims was a flop.  A huge armory was rented, great sums were spent on  advertising the appearance of the black supremacy leader,  Elijah Muhammad, and a minimum of 15,000 persons was  expected; but only about 5,000, 
    mostly Muslims, showed  up.  With over one and a quarter million Negroes in New  York City, this must be regarded as a disappointing turnout for the most publicized Negro organization of the last  decade, considering the hordes that 
    greeted the Negro  Elks at their New York convention.  The sad fact is that Harlemites were conspicuous by  their absence, and those that attended the Muslim meeting gathered outside and listened by loudspeaker.  
    Negro Americans, it is apparent, are not concerned  with queer, unreasonable, extremist organizations that  advance ridiculous programs of salvation, and they are  no more fetched by appeals to black supremacy than programs of white supremacy.  
    In short, black Ku Klux Klans are no more acceptable to Negroes than white Ku Klux Klans.  Views and Reviews  By George S. Schuyler  DESPITE THE current  gnashing of teeth and  frothing at the mouth over colonialism-imperialism 
    (spurred  by the Communists, who  have the largest, and most populous colonies of all), there is  every Indication that it will be  with us for  centuries as it  has been with  us for mlllen-  iums.  ÎÃÃ¶ The so-called backward  and newly-  
    emergent  "democracies"  in Asia and  Africa, along  with the rag-  t a g "republics" of Latin  American derivation, will not only continue  to be dependent upon the big  political power structures, but  will become more dependent.  Whether the free or communist powers prevail, the white  folks, for better or for worse,  will remain far in the lead, to  the great pain, one presumes,  of the militant, black, brown  and yellow nationalists who  yap in the U. N. corridors  and assembly halls.  Mr. Schuyler  WHY, DEAR brethren, is this  so?  Because (a) the underdeveloped nations (largely one-crop  economies just emerging from  subsistence agriculture) are  falling father behind economically, as the mechanized  nations forge farther ahead;  because (b) their populations  continue to swell, thus wiping  out the advantage of any gains  that are made because (c)  modern chemistry has already  made Western industry free of  reliance on the copper, tin, and  bauxite of tropical lands, and  there will be less and less importing of these resources, even  iron which is yielding to synthetics and plastics; because  (d) world supplies of cocoa, oil,  coffee and tea are in such sur.  plus that the price Is everywhere depressed, as is true of  sugar; because (e) it costs at  least an Investment of $600 for  each factory job desired, and  the money is only available  from the free or Communist  power structures, and it is their  bounty and goodwill that holds  the future of these weaker and  poorer countries in their grip.  Thus freedom becomes academic, like the freedom of the  man who is "bugged" by weekly installment payments.  ONE WONDERS how many  of the miseducated leaders of  the backward countries know  these facts of life of the modern world. Every one of these  countries is bankrupt or close  to it.  Having steamed up their few  literate followers to believing  that they have resources that  some body has to buy and  which will pay for their necessary education and industrialization, it must have dawned on  somebody has to buy and  become less indispensable yearly, as substitutes are created in  growing profusion by the great  industrial laboratories, even to  diamonds.  So the pretense that these  scores of weak and improv-  erished countries have the USA  and/or the USSR over the barrel, is a hollow one.  These countries will go on  having their flags, legislatures,  miniscule armies and Cadillac-  ed politicians, but to keep from  being tossed into the pot by  the hungry hordes they have  organized, they will have to  continue to get handouts from  those who will give them. If  this isn't colonialismÎÃÃ¶imperialism, what is it?  A New South Dawning . .    by f. l shuttiesworth  I BELIEVE THAT the South  has learned one lesson well.  That is, that violence, either  by mobs or policemen, will not  be allowed to frustrate the legitimate claims by Negroes of  their rights, no longer. Those  of us who have fought the civil  rights battle  (for years)  against all  odds ... from  frustrated officials to bellicose and enraged neighbors, from per-  secution by  law officers to  outright vici-  ousness by the  Klansmen . . .  have prayed  long for the  dawning of this day.  Always, we have known that  when the men in charge of the  city, hamlet or state, will array themselves, firmly, against  violence and declare their determination to enforce the  laws, the problems will be lessened, and solutions of any difficulties will be found, easily.  Shuttiesworth  Courier  The Pittsburgh Courier Publishing Company, Inc.  2628 Central Ave.  PittsburghTelephone:  MUseum 3-2000  SATURDAY, SEPTEMBER 9, 1961  ROBERT L VANH. EDITOR  MRS.   ROBERT  L.  VANN  president and treasurer  S.   B.  FULLER  Publisher  Chm. of Board  P.   L.   PRATTIS  Associate  Publisher,  Assnt. Treasurer.  WILLIAM A. NUNN   BR.  Editor  Second-rate Postage Paid at Pittsburgh,  Pa.  SUBSCRIPTION RATES  la Continenta   United States,  Hawaii, Alaska  Puerto Rico and Virgin Islands  SIX  MONTHS  $5.00   ... ONE YEAR  $10.00 ... TWO YEARS  IN CANADA  $4.28    SIX  MONTHS  $7.00 ONE YEAR  $12.00 TWO YEARS  FOREIGN  $8.25 SIX MONTHS  ONE YEAR  $18.00 TWO YEARS  The Plttahurgh Coirlar does not guarantee  either the use or return of photographs.  There would have been no  Little Rocks nor New Orleans,  had the Fauduses and Dav-  ises been men fitted for this  hour. But this is history . . .  we hope. And, we pray that the  future pages in this book will  reflect sanity, uprightness and  justice.  ÎÃ»Ã¡  THUS, ALL Americans can  take pride in the programs of  law enforcement, announced  for the Atlanta, Dallas and  Houston schools to be integrated this week. Crowds will not  be allowed to form, and people  with no business at the schools  will not be allowed to enter.  This is a positive step forward, and one in marked contrast to that where crowds  milled and molested everybody  . . . even police officers. The  outcome is clear: law and order will prevail, and the chaos  which, almost, has become traditional will be a thing of the  past.  This is not, by any maens, to  praise token integration . . .  for that is not real American  democracy. And American Negroes never must compromise  their rights, nor be satisfied  with crumbs when they, like  others, are entitled to the loaf.  But, since the integration ordered is token . . . and this is  nothing more than a start, then  it is good to see law officers  guarantee order and peace.  ÎÃ»Ã¡  IT IS MY prayer that the  rulers of Alabama and Birmingham will take lessons from  Mayor Hartsfield, Chief Jenkins and others who represent  progress and enlightenment.  The other side of the coin is  bad. The North Carolina episode, in which Robert Williams  is involved, is unfortunate. So  far as the civil-rights struggle  is concerned, it was wrong to  announce a policy of matching  violence with violence in the  first place. And to kidnap and  hold hostages ... if the press  were right ...   is a think unthinkable!  This played into the hands of  officials who, in the first place,  had no sympathy for the Negro  cause. And, now we read of a  Federal warrant for Mr. Williams' arrest. It may be that  these people were maneuvered  into a trap. We hope for the  best, here.  ÎÃ»Ã¡  JUST THE SAME, the Negro, in his quest for freedom,  MUST use non-violence as a  technique, at least, as the philosophy and need of this hourÎÃÃ¶  in truth. Negroes need the considerate and sympathetic understanding of mankind. Without resort to violence, he has  a better chance of obtaining  it . . . and his full freedom.  Courier Verse  Questions  If you were hungry and had  the cash,  But couldn't even purchase a  plate  of giblet hash,  Because you were beaten by a  Jim Crow lashÎÃÃ¶  Would you like it?  If you were born in a certain  place,  But just because of a colored  face,  You had to live in a special  spaceÎÃÃ¶  Would you like it?  If what you had built someone  tried to destroy,  If you were treated like a child  with a toy,  If strange people termed you  "Uncle" and "Boy"ÎÃÃ¶  Would you like it?  Just questions, America,    we  humbly ask;  Is giving democracy such   a  task?  If freedom and justice wear a  double maskÎÃÃ¶  Should we like it?  ÎÃÃ¶J. FARLEY RAGLAND construed as"
'''

len(check)

12302